# NYC ETA Engine -- Demo

**Dev MAE: 253s** | 3-model ensemble | 28% below baseline

This notebook demonstrates the complete ETA prediction system:
1. Download pre-trained models from HuggingFace
2. Run predictions on sample requests
3. Score on the full dev set
4. Visualize model behavior and ensemble analysis

**No GPU needed** -- inference runs on CPU in <5ms per request.

In [ ]:
# Setup
!git clone https://github.com/sarthakbiswas97/eta-engine.git
%cd eta-engine
!pip install -q -r requirements.txt huggingface_hub onnxruntime

In [ ]:
# Download pre-trained models from HuggingFace
from huggingface_hub import hf_hub_download
import shutil, os

for filename in ["model.pt", "lgbm_model.txt", "ft_model.pt", "ft_model.onnx", "ft_model.onnx.data", "ft_norm_params.npz"]:
    if not os.path.exists(filename):
        path = hf_hub_download("sarthakbiswas/eta-engine", filename)
        shutil.copy2(path, filename)
        print(f"Downloaded {filename}")
    else:
        print(f"Already exists: {filename}")

# Download dev data for scoring
!python data/download_data.py
!PYTHONPATH=. python -m features.zone_pair_stats

---
## 1. Single Request Prediction

The `predict()` function takes a ride request and returns trip duration in seconds.

In [ ]:
import sys
sys.path.insert(0, '.')
from predict import predict
import time

# Sample requests
requests = [
    {"pickup_zone": 236, "dropoff_zone": 237, "requested_at": "2024-02-14T08:30:00", "passenger_count": 1},
    {"pickup_zone": 132, "dropoff_zone": 138, "requested_at": "2024-02-14T17:00:00", "passenger_count": 2},
    {"pickup_zone": 48,  "dropoff_zone": 79,  "requested_at": "2024-12-25T02:00:00", "passenger_count": 1},
    {"pickup_zone": 100, "dropoff_zone": 200, "requested_at": "2024-01-15T12:00:00", "passenger_count": 3},
]

print(f"{'Pickup':>8} {'Dropoff':>8} {'Time':<20} {'Prediction':>12} {'Minutes':>8}")
print("-" * 65)
for req in requests:
    pred = predict(req)
    print(f"{req['pickup_zone']:>8} {req['dropoff_zone']:>8} {req['requested_at']:<20} {pred:>10.0f}s {pred/60:>7.1f}m")

# Latency benchmark
predict(requests[0])  # warmup
t0 = time.perf_counter()
for _ in range(100):
    predict(requests[0])
latency = (time.perf_counter() - t0) / 100 * 1000
print(f"\nLatency: {latency:.1f}ms per request (limit: 200ms)")

---
## 2. Time-of-Day Sensitivity

The model captures how trip duration varies across the day -- rush hour, late night, etc.

In [ ]:
import matplotlib.pyplot as plt

# Same route, different times
hours = list(range(24))
preds_by_hour = []
for h in hours:
    req = {"pickup_zone": 236, "dropoff_zone": 237, "requested_at": f"2024-02-14T{h:02d}:00:00", "passenger_count": 1}
    preds_by_hour.append(predict(req))

plt.figure(figsize=(10, 4))
plt.plot(hours, [p/60 for p in preds_by_hour], 'o-', color='#2196F3', linewidth=2)
plt.axvspan(7, 9, alpha=0.1, color='red', label='AM Rush')
plt.axvspan(16, 19, alpha=0.1, color='orange', label='PM Rush')
plt.xlabel('Hour of Day')
plt.ylabel('Predicted Duration (minutes)')
plt.title('Trip Duration by Time of Day (Zone 236 -> 237)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(hours)
plt.tight_layout()
plt.show()

---
## 3. Dev Set Scoring

Score on a 50k sample of the dev set (same as `python grade.py`).

In [ ]:
import pandas as pd
import numpy as np

dev = pd.read_parquet('data/dev.parquet').sample(n=50000, random_state=42)
print(f"Scoring on {len(dev):,} dev rows...")

preds = []
records = dev[['pickup_zone', 'dropoff_zone', 'requested_at', 'passenger_count']].to_dict('records')
for i, req in enumerate(records):
    preds.append(predict(req))
    if (i+1) % 10000 == 0:
        print(f"  {i+1:,} / {len(dev):,}")

preds = np.array(preds)
truth = dev['duration_seconds'].values
mae = np.mean(np.abs(preds - truth))
bias = np.mean(preds - truth)

print(f"\nDev MAE: {mae:.1f}s")
print(f"Bias: {bias:+.1f}s")
print(f"Baseline (XGBoost): 351s")
print(f"Improvement: {(1 - mae/351)*100:.0f}%")

---
## 4. Ensemble Breakdown

Showing how each model contributes to the ensemble.

In [ ]:
import torch
import lightgbm as lgb
from features.pipeline import FeaturePipeline
from model.architecture import ETAModel, ModelConfig
from model.ft_transformer import FTTransformer, FTConfig, FTConfigSmall
from model.dataset import create_dataloader
import gc

pipeline = FeaturePipeline.from_artifacts('data/zone_pair_stats/zone_pair_stats.pkl')

# Transform dev data
cat, cont, targets = pipeline.transform_dataframe(dev.copy())

# NN predictions
nn_cp = torch.load('model.pt', map_location='cpu', weights_only=False)
nn_model = ETAModel(ModelConfig(**nn_cp['model_config']))
nn_model.load_state_dict(nn_cp['model_state_dict'])
nn_model.eval()
nn_norm = nn_cp['norm_params']
cont_nn = (cont - nn_norm['means']) / nn_norm['stds']
dummy = np.zeros(len(cat), dtype=np.float32)
loader = create_dataloader(cat, cont_nn, dummy, batch_size=16384, shuffle=False, num_workers=0, pin_memory=False)
nn_preds = []
with torch.no_grad():
    for pu, do, c, _ in loader:
        nn_preds.append(nn_model(pu, do, c).cpu().numpy())
nn_preds = np.concatenate(nn_preds)
del nn_model; gc.collect()

# LGBM predictions
lgbm_model = lgb.Booster(model_file='lgbm_model.txt')
X_lgbm = np.hstack([cat.astype(np.float32), cont])
lgbm_preds = lgbm_model.predict(X_lgbm)
del lgbm_model; gc.collect()

# FT predictions
ft_cp = torch.load('ft_model.pt', map_location='cpu', weights_only=False)
ft_cfg = ft_cp['model_config']
if ft_cfg.get('d_token', 128) <= 96:
    ft_model = FTTransformer(FTConfigSmall(**ft_cfg))
else:
    ft_model = FTTransformer(FTConfig(**ft_cfg))
ft_model.load_state_dict(ft_cp['model_state_dict'])
ft_model.eval()
ft_norm = ft_cp['norm_params']
cont_ft = (cont - ft_norm['means']) / ft_norm['stds']
x_num_t = torch.from_numpy(cont_ft).float()
x_cat_t = torch.from_numpy(cat).long()
ft_preds = []
with torch.no_grad():
    for i in range(0, len(x_num_t), 16384):
        ft_preds.append(ft_model(x_num_t[i:i+16384], x_cat_t[i:i+16384]).cpu().numpy())
ft_preds = np.concatenate(ft_preds)
del ft_model; gc.collect()

# Ensemble
ensemble = 0.5 * nn_preds + 0.3 * lgbm_preds + 0.2 * ft_preds

print(f"{'Model':<20} {'MAE':>8} {'Bias':>8}")
print("-" * 40)
for name, p in [("NN (MLP)", nn_preds), ("LightGBM", lgbm_preds), ("FT-Transformer", ft_preds), ("Ensemble", ensemble)]:
    m = np.mean(np.abs(p - targets))
    b = np.mean(p - targets)
    print(f"{name:<20} {m:>8.1f} {b:>+8.1f}")

In [ ]:
# Error distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, p) in zip(axes, [("NN", nn_preds), ("LightGBM", lgbm_preds), ("Ensemble", ensemble)]):
    errors = p - targets
    ax.hist(errors, bins=100, range=(-1000, 1000), alpha=0.7, color='#2196F3')
    ax.axvline(0, color='red', linestyle='--', alpha=0.5)
    ax.axvline(errors.mean(), color='green', linestyle='-', label=f'bias={errors.mean():+.0f}s')
    ax.set_title(f'{name} (MAE={np.mean(np.abs(errors)):.0f}s)')
    ax.set_xlabel('Error (seconds)')
    ax.legend()

plt.suptitle('Prediction Error Distribution', fontsize=14)
plt.tight_layout()
plt.show()

---
## 5. Model Architecture Summary

Three models, ~6 MB total weights, <5ms combined inference.

In [ ]:
print("=" * 60)
print("ETA ENGINE -- ARCHITECTURE SUMMARY")
print("=" * 60)

print("""
Request -> Feature Pipeline (24 features)
              |
    +---------+---------+
    |         |         |
   MLP     LightGBM   FT-Transformer
  (560k)   (81 trees)  (169k, ONNX)
    |         |         |
    +---------+---------+
              |
    0.5*NN + 0.3*LGBM + 0.2*FT
              |
       predicted duration

Features:
  14 zone-pair stats (Bayesian shrinkage, time-bucketed)
  10 temporal features (cyclical, binary flags)
   2 zone IDs (embeddings for NN/FT, categoricals for LGBM)

Constraints:
  Inference: <5ms per request (limit: 200ms)
  Docker: 1.4 GB (limit: 2.5 GB)
  Weights: 6.1 MB total
""")

print("Dev MAE: 253s (28% below 351s baseline)")
print("\nPre-trained models: huggingface.co/sarthakbiswas/eta-engine")